# CSpack Python Kernel Examples

The figures below are cached PNGs, so they are visible as soon as the notebook opens. Run the code cells to rebuild the local shared library and regenerate the plots from the Python wrapper.

## References

- Chluba, Cyr, Uwabo-Niibo, and Yamaguchi, *Neutrino-electron scattering kernels in isotropic media*, arXiv:2607.25436.
- Sarkar, Chluba, and Lee, *Dissecting the Compton scattering kernel I: Isotropic media*, arXiv:1905.00868.

## How to Run This Notebook

The images are already visible because they are saved as PNG files in `python/notebooks/figures`.

To run the examples yourself:

1. Run the `Setup` code cell once.
2. Run any example cell with `Shift-Enter`.
3. Plot cells use `show=True`, which displays the recomputed figure in the notebook.
4. Change plot arguments directly in the notebook cell, for example `plot_photon_approximations(recoil_p0=0.02, doppler_omega0=0.01, show=True)`.
5. For fixed-kernel plots, explicit `omega` grids are checked against `cspack.kernel_bounds` before plotting.
6. To refresh every cached PNG from the terminal, run `make -C python figures` from the main CSpack directory.

## Setup

Run this cell first. It finds the repo root, imports the wrapper, imports the plotting helpers, and builds the local shared library if it is missing.

In [ ]:
from pathlib import Path
import os
import sys
import numpy as np

cwd = Path.cwd().resolve()
for base in (cwd, *cwd.parents):
    if (base / "python" / "notebooks" / "render_figures.py").exists():
        repo_root = base
        break
    if (base / "notebooks" / "render_figures.py").exists():
        repo_root = base.parent
        break
else:
    raise RuntimeError("Could not locate the CSpack repository root.")

python_dir = repo_root / "python"
notebook_dir = python_dir / "notebooks"
os.environ.setdefault("MPLCONFIGDIR", str(python_dir / ".matplotlib"))
sys.path.insert(0, str(python_dir))
sys.path.insert(0, str(notebook_dir))

import cspack
from render_figures import (
    plot_fixed_photon_neutrino,
    plot_thermal_photon_fd,
    plot_thermal_neutrino_fd,
    plot_photon_approximations,
    plot_neutrino_channels,
    plot_scattering_matrix,
    render_all,
)

cspack.ensure_library()

## Calling the Wrapper

The wrapper accepts scalars or NumPy arrays for the kernel variables. These examples use small grids so they run quickly while showing the call signatures.

### Photon Kernel Calls

Use `cspack.kernel` for fixed-momentum photon-electron kernels. The `kind` argument selects `exact`, `recoil`, `doppler`, or `ur`. Use `cspack.kernel_bounds` to get the physical outgoing-energy range before making a grid.

In [ ]:
omega_min, omega_max = cspack.kernel_bounds(omega0=0.1, p0=0.5)
omega = np.linspace(omega_min, omega_max, 8)

doppler_min, doppler_max = cspack.kernel_bounds(omega0=1.0e-2, p0=0.2)
doppler_omega = np.linspace(doppler_min, doppler_max, 10)[1:-1]

photon_exact = cspack.kernel(omega0=0.1, p0=0.5, omega=omega, kind="exact")
photon_doppler = cspack.kernel(
    omega0=1.0e-2,
    p0=0.2,
    omega=doppler_omega,
    kind="doppler",
)

print("omega grid:", omega)
print("exact photon kernel:", photon_exact)
print("Doppler omega grid:", doppler_omega)
print("Doppler approximation:", photon_doppler)

### Thermal Photon Calls

Use `thermal_photon_fd_kernel` for Fermi-Dirac averaged photon kernels. `add_final_state=3` applies electron blocking and photon stimulation.

In [ ]:
thermal_omega = np.logspace(-4, -1, 12)

photon_thermal = cspack.thermal_photon_fd_kernel(
    omega0=1.0e-3,
    omega=thermal_omega,
    theta=0.1,
    mue=-100.0,
    add_final_state=0,
    kind="exact",
)
photon_stimulated = cspack.thermal_photon_fd_kernel(
    omega0=1.0e-3,
    omega=thermal_omega,
    theta=0.1,
    mue=-100.0,
    add_final_state=3,
    kind="exact",
)

print("thermal photon:", photon_thermal)
print("with photon stimulation:", photon_stimulated)

### Neutrino Channel Calls

Use `neutrino_kernel` for fixed-momentum neutrino kernels. The channel name chooses the flavor and electron/positron target.

In [ ]:
omega = np.linspace(0.1, 0.9, 6)
channels = ["nue_e", "nue_p", "nue_ep", "numu_e", "numu_p"]

neutrino_values = {
    channel: cspack.neutrino_kernel(omega0=0.5, p0=1.0, omega=omega, kind=channel)
    for channel in channels
}

for channel, values in neutrino_values.items():
    print(channel, values)

### Thermal Neutrino Calls

Use `thermal_neutrino_kernel` for Fermi-Dirac averaged neutrino kernels. `add_final_state=2` applies electron and neutrino blocking.

In [ ]:
nu_plain = cspack.thermal_neutrino_kernel(
    omega0=1.0e-3,
    omega=thermal_omega,
    theta=0.1,
    mue=-100.0,
    add_final_state=0,
    kind="nue_e",
)
nu_blocked = cspack.thermal_neutrino_kernel(
    omega0=1.0e-3,
    omega=thermal_omega,
    theta=0.1,
    mue=-100.0,
    add_final_state=2,
    kind="nue_e",
)

print("thermal neutrino:", nu_plain)
print("with final-state blocking:", nu_blocked)

## Fixed-Momentum Photon and Neutrino Kernels

Paper-style comparison for `omega_1 = 0.1` and `p = 0.01, 0.05, 0.1, 0.2, 0.5`.

![Fixed-momentum photon and neutrino kernels](figures/fixed_photon_neutrino_kernels.png)

In [ ]:
# Run this cell with Shift-Enter to recompute and display this plot.
plot_fixed_photon_neutrino(
    omega0=0.1,
    p_values=[0.01, 0.05, 0.1, 0.2, 0.5],
    show=True,
)

## Photon Kernels

Thermally averaged photon-electron redistribution for `theta_e = 0.1`, with and without final-state electron blocking plus photon stimulation.

![Thermally averaged photon kernels](figures/thermal_photon_fd_kernels.png)

In [ ]:
# Run this cell with Shift-Enter to recompute and display this plot.
plot_thermal_photon_fd(
    omega1_values=[1e-3, 1e-2, 1e-1, 1.0, 10.0],
    theta=0.1,
    show=True,
)

### Full, Recoil, and Doppler Photon Limits

The approximation curves are shown only over the physical support of the exact kernel.
Change `recoil_omega0`, `recoil_p0`, `doppler_omega0`, or `doppler_p0` here; the plotted `omega` ranges are recomputed automatically. You can also pass `recoil_omega=` or `doppler_omega=` to provide a custom outgoing-energy grid.

![Photon full/recoil/Doppler comparison](figures/photon_exact_recoil_doppler.png)

In [ ]:
# Run this cell with Shift-Enter to recompute and display this plot.
plot_photon_approximations(
    recoil_omega0=0.1,
    recoil_p0=1.0e-2,
    doppler_omega0=1.0e-2,
    doppler_p0=0.2,
    show=True,
)

## Neutrino Kernels

Thermally averaged electron-neutrino redistribution for `theta_e = 0.1`, with and without final-state electron and neutrino blocking.

![Thermally averaged neutrino kernels](figures/thermal_neutrino_fd_kernels.png)

In [ ]:
# Run this cell with Shift-Enter to recompute and display this plot.
plot_thermal_neutrino_fd(
    omega1_values=[1e-3, 1e-2, 1e-1, 1.0, 10.0],
    theta=0.1,
    show=True,
)

### Neutrino Scattering Channels

Fixed-momentum channel comparison for `omega_1 = 0.5` and `p = 1`.

![Neutrino scattering channels](figures/neutrino_scattering_channels.png)

In [ ]:
# Run this cell with Shift-Enter to recompute and display this plot.
plot_neutrino_channels(
    omega0=0.5,
    p0=1.0,
    channels=["nue_e", "nue_p", "nue_ep", "numu_e", "numu_p"],
    show=True,
)

## Scattering Matrix

This uses CSpack's quadrature weights and the explicit photon scattering-matrix wrapper at `theta_e = 0.1`. The left panel shows the full matrix; the right panel pulls out a few rows as ordinary matrix elements.

![Photon scattering matrix elements](figures/scattering_matrix_elements.png)

In [ ]:
x = np.logspace(-3, 1, 48)
weights = cspack.integral_weights(x)
matrix = cspack.scattering_matrix_explicit(
    x,
    theta=0.1,
    weights=weights,
    kernel="exact",
    epsilon=1.0e-8,
)

print("shape:", matrix.shape)
print("max element:", matrix.max())

In [ ]:
# Run this cell with Shift-Enter to recompute and display this plot.
plot_scattering_matrix(theta=0.1, show=True)

## Regenerate All Cached Figures

This cell rewrites every PNG used by the Markdown image cells above.

In [ ]:
# Run this cell with Shift-Enter to rewrite every cached PNG in figures/.
render_all(write_notebook=False)